In [4]:
import requests
import pandas as pd
import time
import os
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

# --- 1. CONFIGURATION ---
API_KEY = "TON_API_KEY_ICI" # Remplacer par ta vraie clé CCData
SYMBOLS = ["BTC", "ETH"]
CURRENCY = "USD"
START_DATE = "2021-03-15 00:00:00"
END_DATE = "2025-12-01 00:00:00"
LIMIT = 2000

# --- CRÉATION D'UNE SESSION ROBUSTE ---
session = requests.Session()
retries = Retry(total=5, backoff_factor=1, status_forcelist=[500, 502, 503, 504])
session.mount('https://', HTTPAdapter(max_retries=retries))

def download_ccdata_histohour(symbol, start_date, end_date, api_key):
    print(f"\n📥 Début du téléchargement CCData (CCCAGG) pour {symbol}/{CURRENCY}...")
    
    start_ts = int(pd.Timestamp(start_date, tz='UTC').timestamp())
    end_ts = int(pd.Timestamp(end_date, tz='UTC').timestamp())
    
    all_data = []
    current_to_ts = end_ts
    headers = {"authorization": f"Apikey {api_key}"}
    
    while current_to_ts >= start_ts:
        url = f"https://min-api.cryptocompare.com/data/v2/histohour?fsym={symbol}&tsym={CURRENCY}&limit={LIMIT}&toTs={current_to_ts}"
        
        try:
            # On utilise session.get au lieu de requests.get
            response = session.get(url, headers=headers, timeout=10)
            
            if response.status_code != 200:
                print(f"❌ Erreur API: {response.status_code}. Pause de 2s...")
                time.sleep(2)
                continue
                
        except requests.exceptions.RequestException as e:
            # En cas de coupure SSL, on attend et on relance sans crasher
            print(f"⚠️ Micro-coupure SSL détectée. Tentative de reconnexion dans 3 secondes...")
            time.sleep(3)
            continue
            
        json_data = response.json()
        
        if json_data.get("Response") == "Error":
            print(f"❌ Erreur CCData: {json_data.get('Message')}")
            break
            
        # Sécurisation de l'extraction des données
        data = json_data.get("Data", {}).get("Data", [])
        
        if not data:
            break
            
        all_data.extend(data)
        oldest_ts_in_batch = data[0]["time"]
        
        if oldest_ts_in_batch <= start_ts:
            break
            
        current_to_ts = oldest_ts_in_batch - 3600 
        time.sleep(0.5) # Anti-spam
        print(f"Téléchargé en reculant jusqu'au : {pd.to_datetime(oldest_ts_in_batch, unit='s', utc=True)}")

    # --- 2. FORMATAGE ---
    if not all_data:
        print("⚠️ Aucune donnée récupérée.")
        return

    df = pd.DataFrame(all_data)
    df['date'] = pd.to_datetime(df['time'], unit='s', utc=True)
    
    if 'volumefrom' in df.columns:
        df = df.rename(columns={'volumefrom': 'volume'})
        
    cols_to_keep = ['date', 'open', 'high', 'low', 'close', 'volume']
    df = df[[c for c in cols_to_keep if c in df.columns]]
    
    df = df.sort_values('date').drop_duplicates('date')
    df = df[(df['date'] >= pd.Timestamp(start_date, tz='UTC')) & (df['date'] < pd.Timestamp(end_date, tz='UTC'))]
    
    # --- 3. EXPORT ---
    output_file = f"{symbol.lower()}_ccdata_1h.parquet"
    df.to_parquet(output_file)
    print(f"✅ Terminé ! {len(df)} lignes sauvegardées dans {output_file}")

# --- LANCEMENT ---
if API_KEY == "f5e3995da653521f5c83e524183cfd178c7ef836b4170a764c73faf29feb7e0a":
    print("⚠️ N'oublie pas de mettre ta vraie API Key CCData dans le script !")
else:
    for sym in SYMBOLS:
        download_ccdata_histohour(sym, START_DATE, END_DATE, API_KEY)


📥 Début du téléchargement CCData (CCCAGG) pour BTC/USD...
Téléchargé en reculant jusqu'au : 2025-09-08 16:00:00+00:00
Téléchargé en reculant jusqu'au : 2025-06-17 07:00:00+00:00
Téléchargé en reculant jusqu'au : 2025-03-25 22:00:00+00:00
Téléchargé en reculant jusqu'au : 2025-01-01 13:00:00+00:00
Téléchargé en reculant jusqu'au : 2024-10-10 04:00:00+00:00
Téléchargé en reculant jusqu'au : 2024-07-18 19:00:00+00:00
Téléchargé en reculant jusqu'au : 2024-04-26 10:00:00+00:00
Téléchargé en reculant jusqu'au : 2024-02-03 01:00:00+00:00
Téléchargé en reculant jusqu'au : 2023-11-11 16:00:00+00:00
Téléchargé en reculant jusqu'au : 2023-08-20 07:00:00+00:00
Téléchargé en reculant jusqu'au : 2023-05-28 22:00:00+00:00
Téléchargé en reculant jusqu'au : 2023-03-06 13:00:00+00:00
Téléchargé en reculant jusqu'au : 2022-12-13 04:00:00+00:00
Téléchargé en reculant jusqu'au : 2022-09-20 19:00:00+00:00
Téléchargé en reculant jusqu'au : 2022-06-29 10:00:00+00:00
Téléchargé en reculant jusqu'au : 2022-04